# Protein–Protein Interface Energetics

**BioPipelines example** — characterising a protein–protein complex: Prodigy predicts the binding affinity from interface contacts, APBS solves the Poisson–Boltzmann equation for the per-chain electrostatics, and PLIP names the specific salt bridges, hydrogen bonds, and hydrophobic contacts that hold the interface together.

[![Documentation](https://img.shields.io/badge/docs-readthedocs-blue)](https://biopipelines.readthedocs.io/en/latest/)
[![Preprint](https://img.shields.io/badge/preprint-bioRxiv-B31B1B)](https://www.biorxiv.org/content/10.64898/2026.03.11.711024v1)

In [ ]:
# Cell 1: Install BioPipelines and micromamba
!git clone https://github.com/locbp-uzh/biopipelines
%cd biopipelines
!pip install -e ".[colab]"
!wget -q https://github.com/mamba-org/micromamba-releases/releases/latest/download/micromamba-linux-64 -O /usr/local/bin/micromamba && chmod +x /usr/local/bin/micromamba

In [ ]:
# Cell 2: Install tools
from biopipelines.pipeline import *
from biopipelines import Prodigy, PLIP, APBS

with Pipeline(project="Setup", job="InstallTools"):
    Prodigy.install()
    PLIP.install()
    APBS.install()

## Cell 3: Interface Energetics Pipeline

Characterises the barnase–barstar complex (PDB 1BRS) — the textbook electrostatically-steered, high-affinity PPI — and ties the pieces together around its interface.

1. **UniProt** annotates the two partners (barnase P00648, barstar P11540) for the record.
2. **PDB** splits 1BRS into its barnase (chain A) and barstar (chain D) chains. 1BRS holds three copies of the complex (A/B/C = barnase, D/E/F = barstar); we work with the A–D pair.
3. **Prodigy** predicts the interface ΔG and Kᴅ from the A–D interfacial contacts (`interface="A D"`), and reports the % of charged residues in the interface.
4. **APBS** solves the Poisson–Boltzmann equation for **each chain separately**, giving per-chain net charge and isoelectric point — barnase is basic (positive), barstar acidic (negative): the complementary charges that steer association.
5. **PLIP** profiles the A–D interface in protein–protein mode (`mode="peptide", chains="D"`), naming the specific salt bridges, hydrogen bonds, and hydrophobic contacts between the two chains — the residue-level interactions behind Prodigy's ΔG and APBS's complementary charges.
6. A **Plot** puts the per-chain net charges side by side (barnase + / barstar −), and Prodigy's ΔG/Kᴅ is the energetics headline — so the electrostatic complementarity is visible right next to the affinity it drives.

All steps run on CPU, so this notebook completes quickly on free Colab.

In [ ]:
# Cell 3: Pipeline
from biopipelines.pipeline import *
from biopipelines import PDB, UniProt, Prodigy, PLIP, APBS, Panda, Plot

with Pipeline(project="BarnaseBarstar", job="InterfaceEnergetics"):
    Resources(memory="8GB", time="1:00:00")

    # Annotate the partners.
    annotation = UniProt(accessions=["P00648", "P11540"])

    # Full complex for the interface calculation; A=barnase, D=barstar.
    complex = PDB("1BRS")
    affinity = Prodigy(structures=complex, interface="A D")   # delta_g_kcal_mol, kd_M, percent_charged_nis

    # Per-chain structures for chain-resolved electrostatics.
    chains = PDB("1BRS", ids="bnbs", chain=["A", "D"], split_chains=True)   # -> bnbs_A (barnase), bnbs_D (barstar)
    electro = APBS(structures=chains, ion_concentration=0.150)              # net_charge / isoelectric_point per chain

    # Residue-level interface contacts (salt bridges, H-bonds, hydrophobics) between A and D.
    interactions = PLIP(structures=complex, mode="peptide", chains="D")

    # Per-chain net charge side by side — the complementary +/− that steers binding.
    Plot(Plot.Bar(data=electro.tables.electrostatics,
                  x="id", y="net_charge",
                  title="Per-chain net charge: barnase (A, +) vs barstar (D, −)",
                  xlabel="Chain", ylabel="Net charge (e)"))

# Interface energetics headline.
affinity.tables.affinity